In [1]:
import torch
import transformers

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
    print(f"GPU 1: {torch.cuda.get_device_name(1)}")
    
print("Transformers:", transformers.__version__)

d:\HPdata\prism_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU count: 2
GPU 0: NVIDIA RTX A5000
GPU 1: NVIDIA RTX A5000
Transformers: 4.53.0


In [ ]:
# Load PRISM model.

from huggingface_hub import login
login("hf_xx")   # add token
from transformers import AutoModel

model = AutoModel.from_pretrained('paige-ai/Prism', trust_remote_code=True)
model = model.to('cuda')
model.eval()
print("✓ PRISM model has been loaded!")

WARNING[XFORMERS]: Need to compile C++ extensions to use all xFormers features.
    Please install xformers properly (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details
A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "d:\HPdata\prism_env\lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
ModuleNotFoundError: No module named 'triton'


✓ PRISM model has been loaded!


In [ ]:
# Load the Virchow model and save the image embedding files

import torch
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from timm.layers import SwiGLUPacked
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import os

print("Virchow is loading...")
model = timm.create_model(
    "hf-hub:paige-ai/Virchow", 
    pretrained=True, 
    mlp_layer=SwiGLUPacked,     
    act_layer=torch.nn.SiLU       
)
model = model.eval()
model = model.to('cuda')
print("✓ Virchow has been loaded!")

transforms = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
print(f"Transform: {transforms}")

image_folder = r'\data' 
output_folder = r'\data\embeddings'

os.makedirs(output_folder, exist_ok=True)

valid_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
image_files = [p for p in Path(image_folder).rglob('*') if p.suffix.lower() in valid_extensions]
print(f"\n{len(image_files)} images found.")

print("\nEmbeddings are being saved...")
for img_path in tqdm(image_files):
    try:
        # Upload the image and apply the transform
        image = Image.open(img_path).convert('RGB')
        image_tensor = transforms(image).unsqueeze(0).to('cuda')
        
        # Embeddings with Virchow
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(image_tensor)
        
        # Distinguish class tokens and patch tokens
        class_token = output[:, 0]  
        patch_tokens = output[:, 1:]    
        
        # Final embedding: class token + mean patch tokens (Required format for PRISM)
        embedding = torch.cat([class_token, patch_tokens.mean(1)], dim=-1)
        embedding = embedding.to(torch.float16)
        
        # Save embeddings
        output_path = os.path.join(output_folder, f"{img_path.stem}_embedding.pth")
        torch.save({
            'embeddings': embedding.squeeze(0).cpu(),  # [2560]
            'image_name': img_path.name,
            'image_path': str(img_path)
        }, output_path)
        
    except Exception as e:
        print(f"\n✗ Hata ({img_path.name}): {e}")
        import traceback
        traceback.print_exc()

print(f"\n✓ All embeddings are saved.")

Virchow is loading...
✓ Virchow has been loaded!
Transform: Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

3309 images found.

Embeddings are being saved...


100%|██████████| 3309/3309 [01:26<00:00, 38.31it/s]


✓ All embeddings are saved.


In [ ]:
"""
EMBEDDING VISUALIZATION FOR PRISM
"""

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
from pathlib import Path

VIRCHOW_EMBEDDING_FOLDER = r"\data\embeddings"
METHOD = "tsne"  # "tsne" or "umap"

print("📂 Virchow embeddings...")
embedding_folder = Path(VIRCHOW_EMBEDDING_FOLDER)
pth_files = sorted(list(embedding_folder.glob("*.pth")))

print(f"   Number of embeddings: {len(pth_files)}")

embeddings = []
labels = []

for i, pth_file in enumerate(pth_files, start=1): 
    data = torch.load(pth_file, map_location='cpu')
    
    if isinstance(data, dict):
        if 'embeddings' in data:
            emb = data['embeddings']
        else:
            emb = data[list(data.keys())[0]]
    else:
        emb = data
    
    if isinstance(emb, torch.Tensor):
        emb = emb.cpu().numpy()
    
    if len(emb.shape) > 1:
        emb = emb[0]  
    
    embeddings.append(emb)
    labels.append(f"{pth_file}")

embeddings = np.array(embeddings)
print(f"\n✓ {len(embeddings)} embeddings are uploaded")
print(f"  Shape: {embeddings.shape}\n")

if METHOD.lower() == "tsne":
    reducer = TSNE(
        n_components=2,
        perplexity=min(5, len(embeddings) - 1),
        max_iter=1000,
        random_state=42,
        metric='cosine'
    )
    embeddings_2d = reducer.fit_transform(embeddings)
    method_name = "t-SNE"
else:
    reducer = umap.UMAP(
        n_neighbors=min(5, len(embeddings) - 1),
        min_dist=0.1,
        n_components=2,
        random_state=42,
        metric='cosine'
    )
    embeddings_2d = reducer.fit_transform(embeddings)
    method_name = "UMAP"

print(f"✓ {method_name} completed\n")

import pandas as pd
df = pd.DataFrame({
    'image_id': range(1, len(embeddings_2d) + 1),
    'image_name': labels,
    'dim1': embeddings_2d[:, 0],
    'dim2': embeddings_2d[:, 1],
    'model': 'PRISM',
    'encoder': 'Virchow',
    'method': method_name
})
df.to_excel(r"\prism_embeddings_2d.xlsx", index=False)


📂 Virchow embeddings...
   Number of embeddings: 3309


C:\Users\PC4\AppData\Local\Temp\ipykernel_22308\2036258284.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(pth_file, map_location='cpu')



✓ 3309 embeddings are uploaded
  Shape: (3309, 2560)

✓ t-SNE completed



In [ ]:
# After image_selection.py code is executed...

# PRISM aggressive sampling test

import torch
import torch.nn.functional as F
from transformers import AutoModel
import pickle
from datetime import datetime
import os
from pathlib import Path

class PRISMAggressiveSampling:
    def __init__(self, model_path='paige-ai/Prism', gpu_id=0):
        self.gpu_id = gpu_id
        self.device = torch.device(f"cuda:{gpu_id}")
        
        print(f"Loading PRISM model on GPU {gpu_id}")
        self.model = AutoModel.from_pretrained(model_path, trust_remote_code=True)
        self.model = self.model.to(self.device)
        self.model.eval()
        print(f"✓ Model loaded")
    
    def generate_with_aggressive_sampling(self, embedding_path, prompt, temperature, 
                                         filepath, method='aggressive'):
        """
        Test different sampling strategies

        Methods:
            'original': Current (boost=0.1)
            'aggressive': Boost=5.0 + Gaussian noise
            'dropout': Variation with random dropout
            'resample': Resample + propagate for each token position
        """
        
        emb_data = torch.load(embedding_path)
        tile_embedding = emb_data['embeddings'].unsqueeze(0).unsqueeze(0).to(self.device)
        
        caption = self.model.tokenize([prompt]).to(self.device)
        
        with torch.autocast('cuda', torch.float16), torch.inference_mode():
            outputs = self.model(input_ids=caption, tile_embeddings=tile_embedding)
        
        base_logits = outputs['logits'].cpu().float()
        
        # Apply temperature
        if temperature > 0:
            scaled_logits = base_logits / temperature
        else:
            scaled_logits = base_logits.clone()
        
        probs = F.softmax(scaled_logits, dim=-1)
        
        # Sampling strategy
        if method == 'original':
            final_logits = self._original_sampling(scaled_logits, probs, temperature)
        
        elif method == 'aggressive':
            final_logits = self._aggressive_sampling(scaled_logits, probs, temperature)
        
        elif method == 'dropout':
            final_logits = self._dropout_sampling(scaled_logits, temperature)
        
        elif method == 'resample':
            final_logits = self._resample_propagate(scaled_logits, probs, temperature)
        
        else:
            raise ValueError(f"Unknown method: {method}")
        
        output_dict = {
            "logits": final_logits.cpu().float() 
        }
        
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "wb") as f:
            pickle.dump(output_dict, f)
        
        return output_dict
    
    def _original_sampling(self, scaled_logits, probs, temperature):
        
        if temperature == 0:
            return scaled_logits.clone()
        
        sampled = scaled_logits.clone()
        for b in range(scaled_logits.size(0)):
            for t in range(scaled_logits.size(1)):
                token_probs = probs[b, t]
                sorted_probs, sorted_indices = torch.sort(token_probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=0)
                keep_mask = cumulative <= 0.9
                keep_mask[0] = True
                
                filtered_probs = torch.where(keep_mask, sorted_probs, torch.zeros_like(sorted_probs))
                filtered_probs = filtered_probs / filtered_probs.sum()
                
                sampled_idx = torch.multinomial(filtered_probs, 1)
                selected_token = sorted_indices[sampled_idx]
                
                sampled[b, t, selected_token] += 0.1  
        
        return sampled
    
    def _aggressive_sampling(self, scaled_logits, probs, temperature):

        if temperature == 0:
            return scaled_logits.clone()
        
        sampled = scaled_logits.clone()
        
        noise_std_multiplier = 0.7
        noise_std = temperature * noise_std_multiplier
        noise = torch.randn_like(sampled) * noise_std
        sampled = sampled + noise
        
        top_p_threshold = 0.9
        
        boost_multiplier = 3.5
        for b in range(scaled_logits.size(0)):
            for t in range(scaled_logits.size(1)):
                token_probs = probs[b, t]
                sorted_probs, sorted_indices = torch.sort(token_probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=0)
                keep_mask = cumulative <= top_p_threshold
                keep_mask[0] = True
                
                filtered_probs = torch.where(keep_mask, sorted_probs, torch.zeros_like(sorted_probs))
                filtered_probs = filtered_probs / filtered_probs.sum()
                
                sampled_idx = torch.multinomial(filtered_probs, 1)
                selected_token = sorted_indices[sampled_idx]
                
                boost = boost_multiplier * temperature
                sampled[b, t, selected_token] += boost
        
        return sampled
    
    def _dropout_sampling(self, scaled_logits, temperature):
        if temperature == 0:
            return scaled_logits.clone()
        
        sampled = scaled_logits.clone()
        
        dropout_rate = min(0.1 * temperature, 0.3)
        
        mask = torch.rand_like(sampled) > dropout_rate
        sampled = sampled * mask.float()
        
        return sampled
    
    def _resample_propagate(self, scaled_logits, probs, temperature):
        if temperature == 0:
            return scaled_logits.clone()
        
        sampled = scaled_logits.clone()
        vocab_size = scaled_logits.size(-1)
        
        for b in range(scaled_logits.size(0)):
            cumulative_effect = torch.zeros(vocab_size)
            
            for t in range(scaled_logits.size(1)):
                token_probs = probs[b, t]
                
                token_probs = token_probs + cumulative_effect.softmax(dim=0) * 0.1
                token_probs = token_probs / token_probs.sum()
                
                sorted_probs, sorted_indices = torch.sort(token_probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=0)
                keep_mask = cumulative <= 0.9
                keep_mask[0] = True
                
                filtered_probs = torch.where(keep_mask, sorted_probs, torch.zeros_like(sorted_probs))
                filtered_probs = filtered_probs / filtered_probs.sum()
                
                sampled_idx = torch.multinomial(filtered_probs, 1)
                selected_token = sorted_indices[sampled_idx]
                
                cumulative_effect[selected_token] += temperature * 2.0
                
                sampled[b, t, selected_token] += temperature * 3.0
        
        return sampled


if __name__ == "__main__":
    generator = PRISMAggressiveSampling(gpu_id=0)
    
    txt_path = "image_list_150.txt"
    embedding_folder = "\data\embeddings"

    with open(txt_path, "r", encoding="utf-8") as f:
        image_names = [line.strip() for line in f if line.strip()]

    embedding_files = sorted(
        [f for f in os.listdir(embedding_folder) if f.endswith("_embedding.pth")]
    )
    filtered_embeddings = [
        f for f in embedding_files
        if any(name in f for name in image_names)
    ]

    prompts = {
        1: ("This is a histopathology slide showing tissue under microscope."+
            "Please analyze the cellular morphology, tissue architecture, and identify whether this shows normal or abnormal pathological changes."+
            "Focus on: 1) Cell types present 2) Tissue organization 3) Any signs of malignancy or benign changes"),
                
        2: ("Examine this histopathology slide and explain each item:"+ 
            "1) Tissue diagnosis. Evaluate the tissue architecture, noting any disruptions or patterns."+
            "2) Assess the overall tissue organization for normality or abnormalities. "+
            "3) If malignant: histologic grade (well, moderately, or poorly differentiated) with justification."+ 
            "4) Features relevant to staging: invasion depth, lymphovascular invasion, margins. "+
            "5) Prognostic indicators: mitotic count, necrosis, specific molecular markers if identifiable."+ 
            "6) Identify cell types present whether the slide appears normal, benignly abnormal, or potentially malignant."),
        
        3: ("Perform a systematic analysis of this histopathology slide, quantify the following features, and synthesize these findings into a diagnostic interpretation:"+
            "1) Predominant cell types and their approximate proportions."+
            "2) Nuclear-to-cytoplasmic ratio (low/moderate/high)."+
            "3) Mitotic figures: absent/rare/frequent."+
            "4) Degree of pleomorphism: mild/moderate/severe."+
            "5) Architectural disruption: none/focal/diffuse."+
            "6) Inflammatory infiltrate: type and density."),
        11: (
            "You are viewing a microscopic image of tissue from a histopathology slide. "
            "Describe the cell morphology and how the tissue is structurally organized, and state whether the findings appear normal or pathological. "
            "Address: 1) Which cell types are visible 2) How the tissue is arranged 3) Whether there are indications of malignant or benign change"
        ),
        12: (
            "Examine this microscopic tissue image and provide a basic morphological assessment. "
            "Comment on the cells observed, the way the tissue is structured, and whether the appearance is consistent with a normal or an abnormal pathological process. "
            "Cover the following points: cell types identified, tissue organization, and any evidence suggestive of malignancy or a benign condition."
        ),
        13: (
            "Look at this histopathology image and assess the tissue at the cellular level. "
            "Report on the morphology of the cells and the overall architecture of the tissue, concluding with an opinion on whether the changes seen are normal or pathological. "
            "Please specifically note: a) cell types you can identify, b) the tissue's structural pattern, c) any features pointing toward malignancy or a benign process."
        ),
        21: (
            "Review this histopathology slide and address the following points in turn: "
            "1) What is the tissue diagnosis? Comment on the tissue architecture and any disruption or recognizable pattern. "
            "2) Is the overall tissue organization normal or abnormal? "
            "3) If a malignancy is present, provide the histologic grade (well, moderately, or poorly differentiated) along with your reasoning. "
            "4) Note any staging-relevant features such as depth of invasion, lymphovascular invasion, or margin status. "
            "5) Mention prognostic indicators including mitotic count, necrosis, or any identifiable molecular markers. "
            "6) State the cell types present and classify the slide as normal, benign but abnormal, or potentially malignant."
        ),
        22: (
            "Analyze this histopathology slide in a structured manner, covering each of the following: "
            "1) Provide a tissue diagnosis, describing the architecture and highlighting any disrupted or patterned regions. "
            "2) Determine whether the tissue organization overall is normal or shows abnormalities. "
            "3) Should malignancy be present, specify the histologic grade (well/moderately/poorly differentiated) and justify your assessment. "
            "4) Identify staging-related features, including invasion depth, lymphovascular involvement, and margin status. "
            "5) List prognostic markers such as mitotic activity, necrosis, and any identifiable molecular indicators. "
            "6) Specify the cell types observed and indicate whether the tissue is normal, benign with abnormalities, or suspicious for malignancy."
        ),
        23: (
            "Please provide an intermediate-level diagnostic evaluation of this histopathology slide by responding to the items below: "
            "1) Tissue diagnosis — comment on architectural integrity, identifying disruptions or notable patterns. "
            "2) Normality assessment — is the tissue organization within normal limits or abnormal? "
            "3) In case of malignancy, report the histologic grade (well, moderately, poorly differentiated) with supporting rationale. "
            "4) Staging features — comment on invasion depth, lymphovascular invasion, and margin involvement. "
            "5) Prognosis-related findings — mitotic count, necrosis, and molecular markers if visible. "
            "6) Cell type identification — classify the slide as normal, benign/abnormal, or potentially malignant."
        ),
	    31: (
            "Conduct a systematic, quantitative evaluation of this histopathology slide and combine your findings into a final diagnostic interpretation. "
            "Specifically report: 1) The dominant cell types and their relative proportions. "
            "2) The nuclear-to-cytoplasmic ratio, categorized as low, moderate, or high. "
            "3) Mitotic activity, categorized as absent, rare, or frequent. "
            "4) The extent of pleomorphism, rated as mild, moderate, or severe. "
            "5) The degree of architectural disruption, described as none, focal, or diffuse. "
            "6) The type and density of any inflammatory infiltrate present."
        ),
        32: (
            "Carry out a quantitative, step-by-step assessment of this histopathology slide, and integrate the results into a single diagnostic conclusion. "
            "Address the following: 1) Identify the predominant cell populations and estimate their proportions. "
            "2) Rate the nuclear-to-cytoplasmic ratio as low, moderate, or high. "
            "3) Characterize mitotic figures as absent, rare, or frequent. "
            "4) Grade the degree of pleomorphism as mild, moderate, or severe. "
            "5) Describe architectural disruption as none, focal, or diffuse. "
            "6) Specify the type and density of inflammatory cell infiltration."
        ),
        33: (
            "Perform a detailed, quantitative analysis of this histopathological image and synthesize your observations into a diagnostic summary. "
            "Your analysis should quantify: 1) The main cell types present and their approximate relative abundance. "
            "2) Nuclear-to-cytoplasmic ratio, classified as low, moderate, or high. "
            "3) The frequency of mitotic figures: absent, rare, or frequent. "
            "4) The severity of cellular pleomorphism: mild, moderate, or severe. "
            "5) The pattern of architectural disruption: none, focal, or diffuse. "
            "6) The nature and density of any inflammatory infiltrate observed."
        ),
    }

    methods = ['aggressive']
    
    for method in methods:
        for kk in [1,2,3]:# [1,2,3,11,12,13,21,22,23,31,32,33]
            prompt = prompts[kk]
            print(f"\n--- PROMPT {kk} ---")
            
            for i, emb_name in enumerate(filtered_embeddings, start=1):
                if i in range(150): #[7,17,31,135,145]: #>= 26 and i <= 100:
                    save_dir = f"\pkl_files\pkl_outputs_PRISM_q{kk}/pkl_output_{i}"
                    Path(save_dir).mkdir(parents=True, exist_ok=True)
                    embedding_path = os.path.join(embedding_folder, emb_name)
                    print(f"  Image {i}: {emb_name[:-14]}")
                    for temperature in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]: 
                        print("temperature: ",temperature)
                        while True:
                            count = sum(
                                1 for f in os.listdir(save_dir)
                                if f.endswith(".pkl") and f"_temp_{temperature}_" in f
                            )
                            if count >= 30:
                                break
                            else:
                                print("test no: ", count+1)
                                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                                filepath = f"{save_dir}/prism_img_{emb_name[:-14]}_temp_{temperature}_{timestamp}.pkl"
                                output = generator.generate_with_aggressive_sampling(
                                    embedding_path=embedding_path,
                                    prompt=prompt,
                                    temperature=temperature,
                                    filepath=filepath,
                                    method=method
                                )                  
    print("\n\nCOMPLETED FOR PRISM!")   

Loading PRISM model on GPU 0
✓ Model loaded

--- PROMPT 1 ---
  Image 1: 0011f470-940b-4c4a-91eb-a8dbe75ae109
temperature:  0.0
test no:  2
test no:  3
test no:  4
test no:  5
test no:  6
test no:  7
test no:  8
test no:  9
test no:  10
test no:  11
test no:  12
test no:  13
test no:  14
test no:  15
test no:  16
test no:  17
test no:  18
test no:  19
test no:  20
test no:  21
test no:  22
test no:  23
test no:  24
test no:  25
test no:  26
test no:  27
test no:  28
test no:  29
test no:  30
temperature:  0.1
test no:  2
test no:  3
test no:  4
test no:  5
test no:  6
test no:  7
test no:  8
test no:  9
test no:  10
test no:  11
test no:  12
test no:  13
test no:  14
test no:  15
test no:  16
test no:  17
test no:  18
test no:  19
test no:  20
test no:  21
test no:  22
test no:  23
test no:  24
test no:  25
test no:  26
test no:  27
test no:  28
test no:  29
test no:  30
temperature:  0.2
test no:  2
test no:  3
test no:  4
test no:  5
test no:  6
test no:  7
test no:  8
test no:  9
te